# Interactive Kelly Explorer

Use this notebook to see how the Kelly bet changes as the inputs change.

The key inputs are:

- `p`: probability of winning
- `b`: net odds paid on a win
- `f`: fraction of bankroll wagered

For a repeated binary bet, the Kelly fraction is:

`f* = (b * p - (1 - p)) / b`

If `f*` is negative, the game has no edge under these assumptions, so the model says not to bet.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import FloatSlider, IntSlider, interact

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
def kelly_fraction(p, b):
    q = 1 - p
    return (b * p - q) / b


def expected_log_growth(f, p, b):
    return p * np.log(1 + b * f) + (1 - p) * np.log(1 - f)


def simulate_many_bankroll_paths(f, p, b, rounds, simulations, starting_bankroll=1.0, seed=42):
    rng = np.random.default_rng(seed)
    wins = rng.random((simulations, rounds)) < p
    multipliers = np.where(wins, 1 + b * f, 1 - f)
    cumulative = np.cumprod(multipliers, axis=1)
    starting_column = np.full((simulations, 1), starting_bankroll)
    return np.column_stack([starting_column, starting_bankroll * cumulative])

## Explore the Kelly Curve

Move `p` and `b` to see how the optimal bet changes.

`b = 1.0` means even-money odds. If you bet $1 and win, you profit $1.

`b = 2.0` means you profit $2 for each $1 bet.

In [ ]:
def plot_kelly_curve(p=0.55, b=1.0):
    raw_kelly = kelly_fraction(p, b)
    practical_kelly = np.clip(raw_kelly, 0, 1)

    fractions = np.linspace(0, 0.99, 400)
    log_growth = expected_log_growth(fractions, p, b)
    compound_growth = np.exp(log_growth) - 1

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(fractions, compound_growth, label="Compound growth rate")
    ax.axhline(0, color="black", linewidth=1)

    if raw_kelly > 0:
        ax.axvline(practical_kelly, color="tab:red", linestyle="--", label=f"Kelly = {practical_kelly:.1%}")
    else:
        ax.text(0.02, 0.95, "No positive edge: Kelly <= 0", transform=ax.transAxes, va="top")

    ax.set_ylim(-0.05, max(0.02, np.nanmax(compound_growth[:200]) * 1.2))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.1%}"))
    ax.set_title("Compound Growth Rate by Bet Fraction")
    ax.set_xlabel("Fraction of bankroll wagered")
    ax.set_ylabel("Compound growth rate per bet")
    ax.legend()
    plt.show()

    print(f"p: {p:.1%}")
    print(f"b: {b:.2f}")
    print(f"Raw Kelly fraction: {raw_kelly:.1%}")
    print(f"Practical non-negative Kelly fraction: {practical_kelly:.1%}")


interact(
    plot_kelly_curve,
    p=FloatSlider(value=0.55, min=0.01, max=0.99, step=0.01, description="Win prob"),
    b=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="Net odds"),
);

## Simulate Different Bet Sizes

This compares half-Kelly, full Kelly, over-Kelly, and your own custom bet size.

The y-axis shows median final bankroll after the selected number of rounds. Median is used because rare giant winners can distort the average.

In [ ]:
def plot_simulated_outcomes(p=0.55, b=1.0, rounds=200, simulations=2000, custom_fraction=0.20):
    raw_kelly = kelly_fraction(p, b)
    full_kelly = np.clip(raw_kelly, 0, 0.99)

    strategies = {
        "0%": 0.0,
        "Half Kelly": 0.5 * full_kelly,
        "Kelly": full_kelly,
        "2x Kelly": min(2 * full_kelly, 0.99),
        "Custom": min(custom_fraction, 0.99),
    }

    labels = []
    medians = []
    p10s = []
    p90s = []

    for label, f in strategies.items():
        paths = simulate_many_bankroll_paths(f, p, b, rounds, simulations, seed=42)
        final_bankrolls = paths[:, -1]
        p10, p50, p90 = np.percentile(final_bankrolls, [10, 50, 90])

        labels.append(f"{label}\n({f:.0%})")
        p10s.append(p10)
        medians.append(p50)
        p90s.append(p90)

    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x, medians, color="tab:blue", alpha=0.75, label="Median final bankroll")
    ax.vlines(x, p10s, p90s, color="black", linewidth=2, label="10th to 90th percentile")
    ax.axhline(1.0, color="black", linewidth=1, linestyle=":", label="Starting bankroll")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title("Typical Final Bankroll by Strategy")
    ax.set_ylabel("Final bankroll")
    ax.legend()
    plt.show()

    print(f"Kelly fraction under these assumptions: {full_kelly:.1%}")


interact(
    plot_simulated_outcomes,
    p=FloatSlider(value=0.55, min=0.01, max=0.99, step=0.01, description="Win prob"),
    b=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="Net odds"),
    rounds=IntSlider(value=200, min=10, max=1000, step=10, description="Rounds"),
    simulations=IntSlider(value=2000, min=100, max=10000, step=100, description="Sims"),
    custom_fraction=FloatSlider(value=0.20, min=0.0, max=0.99, step=0.01, description="Custom f"),
);

## What to Notice

Changing the inputs changes the correct bet size.

- Higher win probability increases Kelly.
- Better odds increase Kelly.
- No edge means Kelly is zero or negative.
- Over-Kelly betting can look exciting, but it often weakens the typical outcome.